### Pre-processing the "fake-and-real-news-dataset" from kaggle

In [7]:
# Libraries
import csv
import pandas as pd
from sklearn.model_selection import train_test_split

In [202]:
fake = pd.read_csv("Dataset/Fake_raw.csv")
true = pd.read_csv("Dataset/True_raw.csv")
# I do not check for NAs because kaggle says there are none.

In [203]:
# Drop subject and date columns (because they have no value for training)
for df in (fake, true):
    for col in ["subject", "date"]:
        if col in df.columns:
            df.drop(columns=col, inplace=True, errors="ignore")

In [206]:
def clean_title(x):
    s = "" if pd.isna(x) else str(x)
    s = s.strip()  # trim surrounding whitespace/newlines
    
    # remove outer straight quotes
    if len(s) >= 2 and s[0] == '"' and s[-1] == '"':
        s = s[1:-1]

    if len(s) >= 2 and s[0] == ''' and s[-1] == ''':
        s = s[1:-1]
    
    # remove leading blanks/tabs
    s = s.lstrip(" \t")
    return s

for df in (fake, true):
    if "title" in df.columns:
        df["title"] = df["title"].apply(clean_title)

In [208]:
# Add status column (fake = 0 and true = 1)
fake["status"] = 0
true["status"] = 1

In [210]:
# Combine both files & shuffle
data = pd.concat([fake, true], ignore_index=True)
data = data.sample(frac=1.0, random_state=42).reset_index(drop=True)

In [212]:
# Split 60% train, 20% val and 20% test (stratified by status)
train_df, temp_df = train_test_split(
    data, test_size=0.4, stratify=data["status"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["status"], random_state=42
)

In [214]:
# Save as JSON
train_df.to_json("train.json", orient="records", force_ascii=False, indent=2)
val_df.to_json("eval.json", orient="records", force_ascii=False, indent=2)
test_df.to_json("test.json", orient="records", force_ascii=False, indent=2)

### Pre-processing the "Fake News" dataset from kaggle

In [98]:
df = pd.read_csv("fake_or_real_news_raw.csv", dtype=str)
df = df[["title", "text", "label"]]
# I do not check for NAs because kaggle says there are none.

In [100]:
print(df.head(10))

                                               title  \
0                       You Can Smell Hillary’s Fear   
1  Watch The Exact Moment Paul Ryan Committed Pol...   
2        Kerry to go to Paris in gesture of sympathy   
3  Bernie supporters on Twitter erupt in anger ag...   
4   The Battle of New York: Why This Primary Matters   
5                                        Tehran, USA   
6  Girl Horrified At What She Watches Boyfriend D...   
7                  ‘Britain’s Schindler’ Dies at 106   
8  Fact check: Trump and Clinton at the 'commande...   
9  Iran reportedly makes new push for uranium con...   

                                                text label  
0  Daniel Greenfield, a Shillman Journalism Fello...  FAKE  
1  Google Pinterest Digg Linkedin Reddit Stumbleu...  FAKE  
2  U.S. Secretary of State John F. Kerry said Mon...  REAL  
3  — Kaydee King (@KaydeeKing) November 9, 2016 T...  FAKE  
4  It's primary day in New York and front-runners...  REAL  
5    \nI’m not an

In [102]:
# Clean data
df["title"] = df["title"].str.strip()
df["text"]  = df["text"].str.strip()

In [104]:
# Map labels to status
label_map = {
    "FAKE": 0,
    "REAL": 1
}
df["status"] = df["label"].map(label_map)
df["status"] = df["status"].astype(int)

In [108]:
# Save as JSON
df[["title", "text", "status"]].to_json(
    "fake_or_real_news.json",
    orient="records",
    force_ascii=False,
    indent=2
)

In [110]:
print("Label distribution:", df["status"].value_counts().to_dict())

Label distribution: {1: 3171, 0: 3164}
